# Supply Chain + News Dataset Merge

**Goal:** Enrich the supply chain logistics dataset with relevant news headlines from the HuffPost News Category Dataset.

**Key design decision:**
- The supply chain dataset spans **2021-01-01 to 2024-08-29**.
- The news dataset (`News_Category_Dataset_v3.json`) covers **2012 to 2022**.
- Since there is no news data for 2023–2024, we use a **month-day (`MM-DD`) fallback merge** — matching news by the same calendar day regardless of year. This is the correct approach and is done explicitly in two steps: first try an exact date match, then fall back to MM-DD.

This notebook is clean and linear — no duplicate merge attempts.

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

Libraries loaded.


## Step 1: Load the Supply Chain Dataset

In [2]:
# Load main supply chain dataset
df = pd.read_csv("dynamic_supply_chain_logistics_dataset.csv")

# Parse timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Create exact date column (YYYY-MM-DD)
df['date_only'] = df['timestamp'].dt.date

# Create month-day column (MM-DD) — used as fallback for news matching
df['month_day'] = df['timestamp'].dt.strftime('%m-%d')

print(f"Shape: {df.shape}")
print(f"Date range: {df['date_only'].min()} → {df['date_only'].max()}")
print()
print(df[['timestamp', 'date_only', 'month_day']].head())

Shape: (32065, 28)
Date range: 2021-01-01 → 2024-08-29

            timestamp   date_only month_day
0 2021-01-01 00:00:00  2021-01-01     01-01
1 2021-01-01 01:00:00  2021-01-01     01-01
2 2021-01-01 02:00:00  2021-01-01     01-01
3 2021-01-01 03:00:00  2021-01-01     01-01
4 2021-01-01 04:00:00  2021-01-01     01-01


## Step 2: Load and Filter the News Dataset

We filter to supply-chain-relevant categories and keywords.

In [3]:
# Load news dataset
news_df = pd.read_json("News_Category_Dataset_v3.json", lines=True)

print(f"News dataset shape: {news_df.shape}")
print(f"News date range: {news_df['date'].min()} → {news_df['date'].max()}")
print(f"Categories: {news_df['category'].nunique()} unique")
print()
print(news_df.head())

News dataset shape: (209527, 6)
News date range: 2012-01-28 00:00:00 → 2022-09-23 00:00:00
Categories: 42 unique

                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johns

In [4]:
# Filter to relevant categories
relevant_categories = ['BUSINESS', 'WORLD NEWS', 'POLITICS']
news_df = news_df[news_df['category'].isin(relevant_categories)]

# Filter to supply-chain-relevant keywords in headlines
keywords = [
    'supply chain', 'logistics', 'shipping', 'transport',
    'inventory', 'warehouse', 'freight', 'delivery', 'cargo'
]
pattern = '|'.join(keywords)

news_filtered = news_df[
    news_df['headline'].str.contains(pattern, case=False, na=False)
].copy()

# Parse date
news_filtered['date'] = pd.to_datetime(news_filtered['date']).dt.date

# Create month_day for fallback matching
news_filtered['month_day'] = pd.to_datetime(news_filtered['date']).dt.strftime('%m-%d')

print(f"Filtered news: {len(news_filtered)} articles")
print(f"Date range: {news_filtered['date'].min()} → {news_filtered['date'].max()}")
print()
print(news_filtered[['date', 'month_day', 'headline', 'category']].head())

Filtered news: 56 articles
Date range: 2013-01-27 → 2022-08-27

            date month_day                                           headline  \
153   2022-08-27     08-27  6 Missing College Students In Mexico Were Held...   
823   2022-04-19     04-19  Justice Department Prepares To Appeal Judge's ...   
861   2022-04-12     04-12  How A Trump Tax Break For The Poor Led To A $3...   
1585  2021-11-27     11-27  Climate Activists Blockade Amazon Warehouses A...   
2235  2021-08-04     08-04  Federal Judge Temporarily Blocks Texas Order R...   

        category  
153   WORLD NEWS  
823     POLITICS  
861     POLITICS  
1585  WORLD NEWS  
2235    POLITICS  


## Step 3: Group News Headlines by Date

We create two lookup tables:
- **`news_by_exact_date`**: grouped by exact date (YYYY-MM-DD) — for rows where the news dataset has coverage
- **`news_by_month_day`**: grouped by MM-DD — for fallback matching when exact year isn't covered (2023–2024 rows)

In [5]:
# Group by exact date
news_by_exact_date = (
    news_filtered
    .groupby('date')['headline']
    .apply(lambda x: ' | '.join(x))
    .reset_index()
    .rename(columns={'headline': 'headline_exact'})
)

# Group by month-day (MM-DD) — combines headlines across all years for that calendar day
news_by_month_day = (
    news_filtered
    .groupby('month_day')['headline']
    .apply(lambda x: ' | '.join(x.unique()))  # deduplicate if same headline appears in multiple years
    .reset_index()
    .rename(columns={'headline': 'headline_monthday'})
)

print(f"Exact-date groups: {len(news_by_exact_date)} unique dates")
print(f"Month-day groups:  {len(news_by_month_day)} unique MM-DD combinations")
print()
print(news_by_exact_date.head())
print()
print(news_by_month_day.head())

Exact-date groups: 55 unique dates
Month-day groups:  52 unique MM-DD combinations

         date                                     headline_exact
0  2013-01-27  The Global Supply Chain: Our Economy, Security...
1  2013-02-09  U.S. Postal Service Right To End Era Of Saturd...
2  2013-07-05  Hostess To Start Freezing Some Twinkies Before...
3  2013-08-11  U.S. Postal Service Alcohol Delivery Idea Crit...
4  2013-12-25  Getting Your Presents On Christmas Involves So...

  month_day                                  headline_monthday
0     01-07  32 Missing As Tanker Collides With Cargo Ship ...
1     01-10  SpaceX Rocket Launches Space Cargo, But Crash ...
2     01-12  Ben Carson Thinks The Government Warehouses Pe...
3     01-15  Pivotal Union Election Moves Ahead At Amazon W...
4     01-22  China Halts Transportation Out Of Wuhan To Sto...


## Step 4: Merge — Exact Date First, then MM-DD Fallback

This is the correct two-step merge strategy:
1. Join on exact `date_only` — works well for 2021–2022 rows
2. For rows that got no match (2023–2024), fill in using the MM-DD fallback

In [6]:
# Step 4a: Exact date merge
merged_df = pd.merge(
    df,
    news_by_exact_date,
    left_on='date_only',
    right_on='date',
    how='left'
)
# Drop the duplicate 'date' column from right side
merged_df.drop(columns=['date'], inplace=True, errors='ignore')

exact_match_count = merged_df['headline_exact'].notna().sum()
print(f"Rows matched by exact date: {exact_match_count:,} / {len(merged_df):,}")

# Step 4b: Month-day fallback merge
merged_df = pd.merge(
    merged_df,
    news_by_month_day,
    on='month_day',
    how='left'
)

# Step 4c: Combine — use exact match if available, else use MM-DD fallback
merged_df['headline'] = merged_df['headline_exact'].combine_first(merged_df['headline_monthday'])

# Fill any remaining NaN (calendar days with no news at all)
merged_df['headline'] = merged_df['headline'].fillna('No relevant news')

# Clean up helper columns
merged_df.drop(columns=['headline_exact', 'headline_monthday'], inplace=True)

print(f"Rows with any headline:     {(merged_df['headline'] != 'No relevant news').sum():,}")
print(f"Rows with 'No relevant news': {(merged_df['headline'] == 'No relevant news').sum():,}")

Rows matched by exact date: 192 / 32,065
Rows with any headline:     4,608
Rows with 'No relevant news': 27,457


## Step 5: Verify Results

In [7]:
# Check early rows (2021) — should have exact matches
print("=== Early rows (2021) ===")
print(merged_df[['timestamp', 'headline']].head(5).to_string())
print()

# Check late rows (2024) — should have MM-DD fallback matches (not NaN)
print("=== Late rows (2024) ===")
print(merged_df[['timestamp', 'headline']].tail(5).to_string())

=== Early rows (2021) ===
            timestamp          headline
0 2021-01-01 00:00:00  No relevant news
1 2021-01-01 01:00:00  No relevant news
2 2021-01-01 02:00:00  No relevant news
3 2021-01-01 03:00:00  No relevant news
4 2021-01-01 04:00:00  No relevant news

=== Late rows (2024) ===
                timestamp          headline
32060 2024-08-28 20:00:00  No relevant news
32061 2024-08-28 21:00:00  No relevant news
32062 2024-08-28 22:00:00  No relevant news
32063 2024-08-28 23:00:00  No relevant news
32064 2024-08-29 00:00:00  No relevant news


In [8]:
# Coverage by year
merged_df['year'] = merged_df['timestamp'].dt.year
coverage = (
    merged_df.groupby('year')
    .apply(lambda g: (g['headline'] != 'No relevant news').mean() * 100)
    .reset_index()
    .rename(columns={0: 'coverage_%'})
)
print("News coverage by year:")
print(coverage.to_string(index=False))

# Drop helper year column
merged_df.drop(columns=['year'], inplace=True)

News coverage by year:
 year  coverage_%
 2021   14.246575
 2022   14.246575
 2023   14.246575
 2024   14.935177


In [9]:
# Final dataset overview
print(f"Final merged_df shape: {merged_df.shape}")
print(f"Columns: {list(merged_df.columns)}")
print()
print(merged_df[['timestamp', 'date_only', 'month_day', 'risk_classification', 'headline']].head(10))

Final merged_df shape: (32065, 29)
Columns: ['timestamp', 'vehicle_gps_latitude', 'vehicle_gps_longitude', 'fuel_consumption_rate', 'eta_variation_hours', 'traffic_congestion_level', 'warehouse_inventory_level', 'loading_unloading_time', 'handling_equipment_availability', 'order_fulfillment_status', 'weather_condition_severity', 'port_congestion_level', 'shipping_costs', 'supplier_reliability_score', 'lead_time_days', 'historical_demand', 'iot_temperature', 'cargo_condition_status', 'route_risk_level', 'customs_clearance_time', 'driver_behavior_score', 'fatigue_monitoring_score', 'disruption_likelihood_score', 'delay_probability', 'risk_classification', 'delivery_time_deviation', 'date_only', 'month_day', 'headline']

            timestamp   date_only month_day risk_classification  \
0 2021-01-01 00:00:00  2021-01-01     01-01       Moderate Risk   
1 2021-01-01 01:00:00  2021-01-01     01-01           High Risk   
2 2021-01-01 02:00:00  2021-01-01     01-01           High Risk   
3 20

## Step 6: Export

In [10]:
# Export to CSV
merged_df.to_csv("merged_supply_chain_with_news.csv", index=False)
print("Saved: merged_supply_chain_with_news.csv")

Saved: merged_supply_chain_with_news.csv
